# Day 2-03｜讓整段影片自動運作：Detection、球場 Keypoint 與 BEV 整合

> 前一單元使用一張畫面與一組固定 Homography；現在我們把流程延伸到影片，讓每個 frame 都重新偵測球員與球場結構。  
> 這是第一次把多個模型與幾何步驟串成 pipeline（處理流程）。

## 我們會完成什麼

- 對每個 frame 執行球員偵測與球場 keypoint 偵測。
- 從當前球場 keypoints 估計 Homography，再投影球員 footpoints。
- 輸出「原始視角 + BEV」左右並排影片與逐 frame JSON。
- 看懂本流程仍缺少跨 frame 身分，因此自然銜接 Day 3 tracking。

## 先認識本單元名詞

- **Pipeline（處理流程）**：前一步輸出成為後一步輸入的一連串運算。
- **Per-frame（逐影格）**：每張畫面獨立處理；此時相鄰畫面中的同一人還沒有共同 ID。
- **Fallback（備援）**：當某一 frame 的 keypoints 不足時，暫時沿用上一個可靠結果以降低閃爍。


## 執行環境提醒

- 建議在 Colab 選擇 **GPU** 執行階段；Ultralytics / PyTorch 不會在本課程設定下直接使用 TPU。
- 沒有 GPU 仍可用 CPU 執行，但模型推論與影片輸出會比較久。先把 `MAX_FRAMES` 調小，就能快速確認流程。
- 每格執行前先讀「這一格要做什麼」，再看輸出是否符合預期；不要只以「沒有紅字」判斷成功。


## 課程流程
1. 選擇另一支參考影片。
2. 對每個 frame 同步執行 detection 與即時場地 keypoint。
3. 將即時場地 keypoint 與 Homography 整合後，把 player footpoint 持續投影到 BEV，產生 `d2_03_detector_keypoint_bev.mp4` 與對應 JSON。


In [1]:
# 這一格要做什麼：定位課程資料夾、必要時取得 repo，並載入共用的課程環境。
# 建議先執行原始設定；確認輸出後，再一次只改一個參數觀察差異。
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [
        Path("/content/drive/MyDrive/basketball_hackathon/course"),  # New: Prioritize Google Drive path
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents
    ] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),  # Fallback if not found anywhere else
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


In [2]:
# 這一格要做什麼：課程流程
# 建議先執行原始設定；確認輸出後，再一次只改一個參數觀察差異。
from src.video_utils import display_video_in_notebook
from src.yolo_utils import preferred_court_keypoint_model_path, write_detector_keypoint_bev_video

videos = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if len(videos) < 2:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要兩支參考影片。")

VIDEO_PATH = videos[1]
DETECTOR_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
COURT_MODEL_PATH = preferred_court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("start frame:", 90)


video: H:\Repos\basketball-hackathon-course\assets\raw\reference_videos\boston-celtics-new-york-knicks-game-1-q1-03.16-03.11.mp4
detector: H:\Repos\basketball-hackathon-course\assets\models\detectors\yolo26n_basketball_player_best.pt
court model: H:\Repos\basketball-hackathon-course\assets\models\court_keypoints\yolo26n_basketball_court_pose_best.pt
start frame: 90


In [3]:
# 這一格要做什麼：課程流程
# 建議先執行原始設定；確認輸出後，再一次只改一個參數觀察差異。
bev_video = COURSE_ROOT / "assets" / "results" / "d2_03_detector_keypoint_bev.mp4"
bev_video, rows = write_detector_keypoint_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=bev_video,
    max_frames=90,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=90,
)
bev_json = bev_video.with_suffix(".json")

print("BEV video:", bev_video)
print("BEV json:", bev_json)
print("projected rows:", len(rows))
display_video_in_notebook(bev_video, loop=True)


$ C:\Users\henry\scoop\apps\ffmpeg-shared\8.1\bin\ffmpeg.exe -y -i H:\Repos\basketball-hackathon-course\assets\results\d2_03_detector_keypoint_bev.mp4 -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart -preset veryfast -crf 23 H:\Repos\basketball-hackathon-course\assets\results\d2_03_detector_keypoint_bev.notebook.mp4
BEV video: H:\Repos\basketball-hackathon-course\assets\results\d2_03_detector_keypoint_bev.mp4
BEV json: H:\Repos\basketball-hackathon-course\assets\results\d2_03_detector_keypoint_bev.json
projected rows: 589


本單元輸出的 BEV 點位尚未維持跨 frame 身分；Day 3 會加入 ByteTrack，讓同一位球員在不同 frame 中保留穩定 track ID。

## 本單元產出檔案

- `assets/results/d2_03_detector_keypoint_bev.mp4`：detector + court keypoint + BEV 並排影片。
- `assets/results/d2_03_detector_keypoint_bev.json`：每個 frame 的 player footpoints 與 BEV 投影資料。
